# ✈️ Aviation VQA — CNN Only (ResNet50 Pixel Reading)
No CLIP. No ChromaDB. Pure CNN pixel reading → answer.

> **Before starting:** Runtime → Change runtime type → **T4 GPU**

## Cell 0 — Run First Every Session

In [ ]:
# ── PATCH Cell 0 in the notebook inside the zip ──────────────────
# Run this ONCE in any Colab cell, then re-download the fixed zip
import zipfile, json, os, shutil

BASE_DIR = "/content/drive/MyDrive/aviation_vqa_output"
ZIP_SRC  = os.path.join(BASE_DIR, "aviation_cnn.zip")
ZIP_DST  = os.path.join(BASE_DIR, "aviation_cnn_fixed.zip")

# Extract zip
EXTRACT_DIR = "/content/aviation_cnn_patch"
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
with zipfile.ZipFile(ZIP_SRC, "r") as z:
    z.extractall(EXTRACT_DIR)
print("Extracted ✓")

# Fix all __init__.py files
MODULES = ["models","data_collection","radar_generation",
           "qa_generation","preprocessing","evaluation",
           "voice","gui"]
for mod in MODULES:
    path = os.path.join(EXTRACT_DIR, "aviation_cnn", mod, "__init__.py")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(f"# {mod}\n")
    print(f"Fixed: {path}")

# Fix Cell 0 in the notebook
NB_PATH = os.path.join(EXTRACT_DIR,
    "aviation_cnn/notebooks/Aviation_CNN_Colab.ipynb")
with open(NB_PATH) as f:
    nb = json.load(f)

# Find Cell 0 and replace it
new_cell0_code = '''\
# ════════════════════════════════════════════════
# CELL 0 — RUN THIS FIRST EVERY SESSION
# ════════════════════════════════════════════════
import os, sys, zipfile, json, shutil

# 1. Mount Drive
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/aviation_vqa_output"
PROC_DIR = os.path.join(BASE_DIR, "data/processed")
RAW_DIR  = os.path.join(BASE_DIR, "data/raw")
print(f"BASE_DIR: {BASE_DIR}")

# 2. Remove old broken extraction
if os.path.exists("/content/aviation_cnn"):
    shutil.rmtree("/content/aviation_cnn")
    print("Removed old extraction")

# 3. Find and extract zip
zip_locations = [
    "/content/aviation_cnn.zip",
    os.path.join(BASE_DIR, "aviation_cnn.zip"),
    "/content/drive/MyDrive/aviation_cnn.zip",
]
found = None
for p in zip_locations:
    if os.path.exists(p):
        found = p
        break

if found:
    print(f"Found zip: {found}")
    with zipfile.ZipFile(found, "r") as z:
        z.extractall("/content/")
    print("Extracted ✓")
else:
    from google.colab import files
    print("Upload aviation_cnn.zip now:")
    uploaded = files.upload()
    for fname in uploaded:
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall("/content/")
        dst = os.path.join(BASE_DIR, "aviation_cnn.zip")
        if not os.path.exists(dst):
            shutil.copy(fname, dst)
    print("Extracted ✓")

# 4. FIX __init__.py — write actual content not empty files
MODULES = ["models","data_collection","radar_generation",
           "qa_generation","preprocessing","evaluation",
           "voice","gui"]
for mod in MODULES:
    folder = f"/content/aviation_cnn/{mod}"
    init   = f"{folder}/__init__.py"
    os.makedirs(folder, exist_ok=True)
    with open(init, "w") as f:
        f.write(f"# {mod}\\n")
print("__init__.py files fixed ✓")

# 5. Clean sys.path — add once only
sys.path = [p for p in sys.path
            if "/content/aviation_cnn" not in p]
sys.path.insert(0, "/content/aviation_cnn")
print(f"sys.path set ✓")

# 6. Load raw_data if exists
raw_manifest = os.path.join(RAW_DIR, "raw_frames.json")
if os.path.exists(raw_manifest):
    with open(raw_manifest) as f:
        raw_data = json.load(f)
    print(f"raw_data: {len(raw_data)} frames ✓")
else:
    raw_data = []
    print("raw_data not found — Step 3 will generate it")

# 7. Test all imports
print("\\nTesting imports...")
tests = [
    ("data_collection.opensky_collector","SyntheticOpenSkyCollector"),
    ("radar_generation.radar_renderer",  "RadarRenderer"),
    ("qa_generation.qa_generator",       "RadarQAGenerator"),
    ("preprocessing.dataset_builder",    "DatasetBuilder"),
    ("models.cnn_model",                 "CNNVQAModel"),
    ("models.cnn_dataset",               "CNNVQADataset"),
    ("models.cnn_trainer",               "CNNVQATrainer"),
    ("models.cnn_inference",             "CNNInferenceEngine"),
    ("evaluation.evaluator",             "CNNEvaluator"),
    ("voice.voice_query",                "TextQueryInterface"),
    ("gui.cnn_demo",                     "CNNPilotDemo"),
]
ok, fail = [], []
for module, cls in tests:
    try:
        mod = __import__(module, fromlist=[cls])
        getattr(mod, cls)
        ok.append(module)
        print(f"  ✓ {module}")
    except Exception as e:
        fail.append(module)
        print(f"  ✗ {module} — {e}")

print(f"\\n{len(ok)}/{len(tests)} imports OK")
if not fail:
    print("All good ✓ Proceed to Step 1")
else:
    print(f"Failed: {fail}")
'''

# Replace Cell 0 in notebook
for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "code":
        src = "".join(cell["source"])
        if "CELL 0" in src or "RUN THIS FIRST" in src:
            nb["cells"][i]["source"] = [new_cell0_code]
            print(f"Patched Cell 0 (index {i}) ✓")
            break

# Save patched notebook
with open(NB_PATH, "w") as f:
    json.dump(nb, f, indent=1)
print("Notebook saved ✓")

# Rezip everything
print("Rezipping...")
with zipfile.ZipFile(ZIP_DST, "w", zipfile.ZIP_DEFLATED) as zout:
    for root, dirs, files in os.walk(EXTRACT_DIR):
        dirs[:] = [d for d in dirs if "__pycache__" not in d]
        for file in files:
            if file.endswith(".pyc"):
                continue
            full_path = os.path.join(root, file)
            arc_name  = os.path.relpath(full_path, EXTRACT_DIR)
            zout.write(full_path, arc_name)

# Replace old zip with fixed one
shutil.copy(ZIP_DST, ZIP_SRC)
shutil.rmtree(EXTRACT_DIR)
os.remove(ZIP_DST)
print(f"Fixed zip saved to Drive: {ZIP_SRC}")
print("\nDone ✓")
print("Now Runtime → Restart runtime")
print("Then run Cell 0 again — all 11 imports will pass")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/aviation_vqa_output/aviation_cnn.zip'

In [2]:
# ════════════════════════════════════════════════
# CELL 0 — RUN THIS FIRST EVERY SESSION
# ════════════════════════════════════════════════
import os, sys, zipfile, json, shutil

# 1. Mount Drive
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/aviation_vqa_output"
PROC_DIR = os.path.join(BASE_DIR, "data/processed")
RAW_DIR  = os.path.join(BASE_DIR, "data/raw")
print(f"BASE_DIR: {BASE_DIR}")

# 2. Remove old broken extraction
if os.path.exists("/content/aviation_cnn"):
    shutil.rmtree("/content/aviation_cnn")

# 3. Find and extract zip
zip_locations = [
    "/content/aviation_cnn.zip",
    os.path.join(BASE_DIR, "aviation_cnn.zip"),
    "/content/drive/MyDrive/aviation_cnn.zip",
]
found = None
for p in zip_locations:
    if os.path.exists(p):
        found = p
        break

if found:
    print(f"Found zip: {found}")
    with zipfile.ZipFile(found, "r") as z:
        z.extractall("/content/")
    print("Extracted ✓")
else:
    from google.colab import files
    print("Upload aviation_cnn.zip now:")
    uploaded = files.upload()
    for fname in uploaded:
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall("/content/")
    # Save to Drive for next session
    for fname in uploaded:
        dst = os.path.join(BASE_DIR, "aviation_cnn.zip")
        if not os.path.exists(dst):
            shutil.copy(fname, dst)
            print(f"Zip saved to Drive ✓")
    print("Extracted ✓")

# 4. Fix __init__.py
MODULES = ["data_collection","radar_generation","qa_generation",
           "preprocessing","models","evaluation","voice","gui"]
for mod in MODULES:
    path = f"/content/aviation_cnn/{mod}/__init__.py"
    if not os.path.exists(path):
        with open(path, "w") as f: f.write(f"# {mod}\n")

# 5. Clean sys.path
sys.path = [p for p in sys.path if "/content/aviation_cnn" not in p]
sys.path.insert(0, "/content/aviation_cnn")

# 6. Load raw_data if exists
raw_manifest = os.path.join(RAW_DIR, "raw_frames.json")
if os.path.exists(raw_manifest):
    with open(raw_manifest) as f:
        raw_data = json.load(f)
    print(f"raw_data: {len(raw_data)} frames ✓")
else:
    raw_data = []
    print("raw_data not found — Step 3 will generate it")

# 7. Test imports
print("\nTesting imports...")
tests = [
    ("data_collection.opensky_collector","SyntheticOpenSkyCollector"),
    ("radar_generation.radar_renderer","RadarRenderer"),
    ("qa_generation.qa_generator","RadarQAGenerator"),
    ("preprocessing.dataset_builder","DatasetBuilder"),
    ("models.cnn_model","CNNVQAModel"),
    ("models.cnn_dataset","CNNVQADataset"),
    ("models.cnn_trainer","CNNVQATrainer"),
    ("models.cnn_inference","CNNInferenceEngine"),
    ("evaluation.evaluator","CNNEvaluator"),
    ("voice.voice_query","TextQueryInterface"),
    ("gui.cnn_demo","CNNPilotDemo"),
]
ok, fail = [], []
for module, cls in tests:
    try:
        mod = __import__(module, fromlist=[cls])
        getattr(mod, cls)
        ok.append(module)
        print(f"  ✓ {module}")
    except Exception as e:
        fail.append(module)
        print(f"  ✗ {module} — {e}")

print(f"\n{len(ok)}/{len(tests)} imports OK")
if not fail:
    print("All good ✓ Proceed to Step 1")
else:
    print("Fix failed imports before continuing")


Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/aviation_vqa_output
Found zip: /content/drive/MyDrive/aviation_vqa_output/aviation_cnn.zip
Extracted ✓
raw_data: 2000 frames ✓

Testing imports...
  ✓ data_collection.opensky_collector
  ✓ radar_generation.radar_renderer
  ✓ qa_generation.qa_generator
  ✓ preprocessing.dataset_builder
  ✓ models.cnn_model
  ✓ models.cnn_dataset
  ✓ models.cnn_trainer
  ✓ models.cnn_inference
  ✓ evaluation.evaluator
  ✓ voice.voice_query
  ✓ gui.cnn_demo

11/11 imports OK
All good ✓ Proceed to Step 1


## Step 1 — Install Packages

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip"] + list(args),
                          stdout=subprocess.DEVNULL,
                          stderr=subprocess.DEVNULL)

print("Installing packages...")
pip("install", "-q",
    "torch", "torchvision", "requests",
    "pillow", "matplotlib", "numpy", "pandas",
    "scikit-learn", "sounddevice",
    "SpeechRecognition", "scipy",
    "tqdm", "ipywidgets")

import torch
print(f"PyTorch : {torch.__version__} ✓")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print("Packages ready ✓")


Installing packages...
PyTorch : 2.11.0+cu128 ✓
CUDA    : True
GPU     : Tesla T4
Packages ready ✓


## Step 2 — Setup Directories

In [ ]:
import os, json, torch

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
PROC_DIR = os.path.join(BASE_DIR, "data/processed")
RAW_DIR  = os.path.join(BASE_DIR, "data/raw")
IMG_DIR  = os.path.join(BASE_DIR, "data/raw/images")

for d in ["data/raw/images","data/processed/images",
          "data/annotations","models",
          "evaluation_results","demo_sessions"]:
    os.makedirs(os.path.join(BASE_DIR, d), exist_ok=True)

print(f"Device   : {DEVICE}")
print(f"BASE_DIR : {BASE_DIR}")
print("Directories ready ✓")


Device   : cuda
BASE_DIR : /content/drive/MyDrive/aviation_vqa_output
Directories ready ✓


## Step 3 — Collect 2000 NEW Frames (Total = 4000, No Duplicates)

In [ ]:
import os, sys, json, hashlib
sys.path.insert(0, "/content/aviation_cnn")

RAW_DIR      = os.path.join(BASE_DIR, "data/raw")
OLD_MANIFEST = os.path.join(RAW_DIR, "raw_frames.json")
NEW_MANIFEST = os.path.join(RAW_DIR, "all_frames.json")

# Load existing 2000 frames
if os.path.exists(OLD_MANIFEST):
    with open(OLD_MANIFEST) as f:
        existing_frames = json.load(f)
    print(f"Existing frames: {len(existing_frames)}")
else:
    existing_frames = []
    print("No existing frames found")

# Check if we already have 4000
if os.path.exists(NEW_MANIFEST):
    with open(NEW_MANIFEST) as f:
        all_frames = json.load(f)
    print(f"Loaded {len(all_frames)} total frames from Drive ✓")
    raw_data = all_frames
else:
    start_idx = len(existing_frames)
    print(f"Collecting 2000 NEW frames (starting at frame_{start_idx:04d}) ...")

    USE_SYNTHETIC = False
    if not USE_SYNTHETIC:
        try:
            import requests
            r = requests.get(
                "https://opensky-network.org/api/states/all",
                params={"lamin":36,"lomin":-10,"lamax":60,"lomax":25},
                timeout=10)
            r.raise_for_status()
            from data_collection.opensky_collector import OpenSkyCollector
            collector = OpenSkyCollector(
                output_dir=RAW_DIR,
                num_new_frames=2000,
                existing_frames=existing_frames,
                aircraft_per_frame=(4,10),
                sleep_between=2.0)
            print("Using OpenSky API ...")
        except Exception as e:
            print(f"OpenSky unavailable ({e}). Using synthetic.")
            USE_SYNTHETIC = True

    if USE_SYNTHETIC:
        from data_collection.opensky_collector import SyntheticOpenSkyCollector
        collector = SyntheticOpenSkyCollector(
            output_dir=RAW_DIR,
            num_new_frames=2000,
            existing_frames=existing_frames,
            aircraft_per_frame=(4,10),
            sleep_between=0.0)

    new_frames = collector.collect(start_idx=start_idx)
    all_frames = existing_frames + new_frames

    # Verify NO duplicates across all 4000
    fps = set()
    dups = 0
    for fr in all_frames:
        key = ",".join(sorted(
            f"{a['latitude']:.2f},{a['longitude']:.2f}"
            for a in fr["aircraft"]))
        fp = hashlib.md5(key.encode()).hexdigest()
        if fp in fps: dups += 1
        fps.add(fp)

    print(f"\nTotal frames : {len(all_frames)}")
    print(f"Duplicates   : {dups} (should be 0)")
    assert dups == 0, "Duplicates found!"

    with open(NEW_MANIFEST, "w") as f:
        json.dump(all_frames, f, indent=2)
    print(f"Saved {len(all_frames)} frames ✓")
    raw_data = all_frames


Existing frames: 2000
OpenSky unavailable (HTTPSConnectionPool(host='opensky-network.org', port=443): Max retries exceeded with url: /api/states/all?lamin=36&lomin=-10&lamax=60&lomax=25 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7b88688107d0>, 'Connection to opensky-network.org timed out. (connect timeout=10)'))). Using synthetic.

Total frames : 4000
Duplicates   : 0 (should be 0)
Saved 4000 frames ✓


## Step 4 — Render 2000 NEW Radar Images (Total = 4000 images)

In [ ]:
import os, sys, json
sys.path.insert(0, "/content/aviation_cnn")
from radar_generation.radar_renderer import RadarRenderer
from PIL import Image
import matplotlib.pyplot as plt, random

IMG_DIR  = os.path.join(BASE_DIR, "data/raw/images")
REN_MAN  = os.path.join(BASE_DIR, "data/raw/rendered_frames.json")

renderer = RadarRenderer(output_dir=IMG_DIR, image_size=224)

if os.path.exists(REN_MAN):
    with open(REN_MAN) as f:
        rendered = json.load(f)
    # Check for new frames not yet rendered
    rendered_ids = {fr["frame_id"] for fr in rendered}
    new_to_render = [fr for fr in raw_data
                     if fr["frame_id"] not in rendered_ids]
    if new_to_render:
        print(f"Rendering {len(new_to_render)} new frames...")
        # Attach existing paths first
        renderer.attach_paths(rendered, IMG_DIR)
        new_rendered = renderer.render_all(new_to_render)
        rendered = rendered + new_rendered
        with open(REN_MAN, "w") as f:
            json.dump(rendered, f, indent=2)
        print(f"Total rendered: {len(rendered)} ✓")
    else:
        print(f"All {len(rendered)} frames already rendered ✓")
else:
    print(f"Rendering all {len(raw_data)} frames...")
    # Attach existing raw images if already there
    existing_rendered = renderer.attach_paths(
        [fr for fr in raw_data
         if os.path.exists(
             os.path.join(IMG_DIR, f"{fr['frame_id']}.png"))],
        IMG_DIR)
    rendered_ids = {fr["frame_id"] for fr in existing_rendered}
    to_render = [fr for fr in raw_data
                 if fr["frame_id"] not in rendered_ids]
    new_rendered = renderer.render_all(to_render)
    rendered = existing_rendered + new_rendered
    with open(REN_MAN, "w") as f:
        json.dump(rendered, f, indent=2)
    print(f"Rendered {len(rendered)} images ✓")

# Preview
samples = random.sample(rendered, min(4, len(rendered)))
fig, axes = plt.subplots(1, len(samples),
                          figsize=(4*len(samples), 4))
if len(samples)==1: axes=[axes]
for ax, fr in zip(axes, samples):
    ax.imshow(Image.open(fr["image_path"]))
    ax.set_title(f"{fr['frame_id']}\n"
                 f"{len(fr['aircraft'])} aircraft", fontsize=8)
    ax.axis("off")
plt.suptitle(f"Sample Radar Images (Total: {len(rendered)})")
plt.tight_layout(); plt.show()


Rendering 2000 new frames...
Total rendered: 4000 ✓


## Step 5 — Generate QA Pairs (~28,000 total for 4000 images)

In [ ]:
import os, sys, json
sys.path.insert(0, "/content/aviation_cnn")
from qa_generation.qa_generator import RadarQAGenerator
from collections import Counter

ANNO_PATH = os.path.join(BASE_DIR,
    "data/annotations/annotations.jsonl")

if os.path.exists(ANNO_PATH):
    with open(ANNO_PATH) as f:
        annotations = [json.loads(l) for l in f if l.strip()]
    print(f"Loaded {len(annotations)} existing QA pairs")

    # Check if new frames need QA
    existing_image_ids = {r["image_id"] for r in annotations}
    new_frames_for_qa = [
        fr for fr in rendered
        if fr["frame_id"] not in existing_image_ids]

    if new_frames_for_qa:
        print(f"Generating QA for {len(new_frames_for_qa)} new frames...")
        qa_gen = RadarQAGenerator()
        new_qa = qa_gen.generate_all(
            frames=new_frames_for_qa,
            qa_per_image=7,
            output_path=ANNO_PATH,
            append=True)
        annotations = annotations + new_qa
        print(f"Total QA pairs: {len(annotations)} ✓")
    else:
        print("All frames have QA pairs ✓")
else:
    print(f"Generating QA for {len(rendered)} frames...")
    for fr in rendered:
        if "image_path" not in fr:
            fr["image_path"] = os.path.join(
                IMG_DIR, f"{fr['frame_id']}.png")
    qa_gen = RadarQAGenerator()
    annotations = qa_gen.generate_all(
        frames=rendered, qa_per_image=7,
        output_path=ANNO_PATH)
    print(f"Generated {len(annotations)} QA pairs ✓")

dist = Counter(r["question_type"] for r in annotations)
print(f"\nTotal: {len(annotations):,} QA pairs")
print("\nQuestion type distribution:")
for t, c in sorted(dist.items(), key=lambda x:-x[1]):
    print(f"  {t:<30} {c:>6}")


Loaded 14000 existing QA pairs
Generating QA for 2000 new frames...
Total QA pairs: 28000 ✓

Total: 28,000 QA pairs

Question type distribution:
  comparative_highest              2044
  region_density                   2044
  heading                          2033
  threshold                        2031
  comparative_lowest               2014
  positional_top                   2011
  altitude_lowest                  2011
  positional_center                1998
  positional_left                  1994
  boolean_count                    1990
  positional_bottom                1982
  altitude_value                   1968
  positional_right                 1941
  counting                         1939


In [ ]:
# Check exactly how many images have QA pairs
import os, json
from collections import Counter

ANNO_PATH = os.path.join(BASE_DIR, "data/annotations/annotations.jsonl")
REMAPPED  = os.path.join(BASE_DIR, "data/annotations/annotations_remapped.jsonl")

# Check which file exists
if os.path.exists(REMAPPED):
    path = REMAPPED
elif os.path.exists(ANNO_PATH):
    path = ANNO_PATH
else:
    print("No annotations found")
    path = None

if path:
    with open(path) as f:
        records = [json.loads(l) for l in f if l.strip()]

    image_ids = set(r["image_id"] for r in records)
    print(f"Total QA pairs       : {len(records):,}")
    print(f"Unique images with QA: {len(image_ids)}")
    print(f"QA pairs per image   : {len(records)/max(len(image_ids),1):.1f}")

    # Show which frame IDs are covered
    frame_nums = sorted([int(fid.replace('frame_',''))
                         for fid in image_ids])
    print(f"Frame range covered  : frame_{min(frame_nums):04d}"
          f" → frame_{max(frame_nums):04d}")
    print(f"\nExpected total images: 4,000")
    print(f"Images missing QA    : {4000 - len(image_ids)}")

Total QA pairs       : 28,000
Unique images with QA: 4000
QA pairs per image   : 7.0
Frame range covered  : frame_0000 → frame_3999

Expected total images: 4,000
Images missing QA    : 0


In [ ]:
# ══════════════════════════════════════════════════════════════════
# DATASET AUGMENTATION CELL
# Run after Step 5, before Fix Cell 1
# Augments all 4000 images → saves new augmented images to Drive
# Total images after: 4000 original + 12000 augmented = 16000
# Total QA pairs after: 28000 + 84000 = 112000
# ══════════════════════════════════════════════════════════════════
import os, sys, json
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import random
sys.path.insert(0, "/content/aviation_cnn")

for mod in ["models","data_collection","radar_generation",
            "qa_generation","preprocessing","evaluation",
            "voice","gui"]:
    p = f"/content/aviation_cnn/{mod}/__init__.py"
    if not os.path.exists(p):
        os.makedirs(os.path.dirname(p), exist_ok=True)
        open(p,"w").write(f"# {mod}\n")

IMG_DIR      = os.path.join(BASE_DIR, "data/raw/images")
AUG_IMG_DIR  = os.path.join(BASE_DIR, "data/augmented/images")
AUG_ANNO     = os.path.join(BASE_DIR, "data/annotations/annotations_augmented.jsonl")
ORIG_ANNO    = os.path.join(BASE_DIR, "data/annotations/annotations.jsonl")
os.makedirs(AUG_IMG_DIR, exist_ok=True)

# ── Load original annotations ─────────────────────────────────────
with open(ORIG_ANNO) as f:
    orig_records = [json.loads(l) for l in f if l.strip()]
print(f"Original QA pairs   : {len(orig_records):,}")
print(f"Original images     : {len(set(r['image_id'] for r in orig_records)):,}")

# ── Define 3 augmentation functions ──────────────────────────────
# Each function takes a PIL image and returns augmented PIL image

def augment_v1(img):
    """
    Augmentation 1 — Geometric transforms
    - Horizontal + vertical flip
    - Random rotation 0-360 degrees
    Reason: Radar has no fixed orientation.
    Aircraft can appear anywhere on the display.
    """
    # Random horizontal flip
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    # Random vertical flip
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    # Full 360 degree rotation
    angle = random.uniform(0, 360)
    img   = img.rotate(angle, resample=Image.BILINEAR,
                       fillcolor=(0, 26, 0))
    return img

def augment_v2(img):
    """
    Augmentation 2 — Photometric transforms
    - Brightness variation ±30%
    - Contrast variation ±30%
    - Gaussian blur (simulates radar noise)
    Reason: Different radar systems have different
    brightness/contrast settings and signal noise levels.
    """
    # Brightness
    factor = random.uniform(0.7, 1.3)
    img    = ImageEnhance.Brightness(img).enhance(factor)
    # Contrast
    factor = random.uniform(0.7, 1.3)
    img    = ImageEnhance.Contrast(img).enhance(factor)
    # Gaussian blur (30% chance)
    if random.random() > 0.7:
        img = img.filter(ImageFilter.GaussianBlur(
            radius=random.uniform(0.5, 1.5)))
    return img

def augment_v3(img):
    """
    Augmentation 3 — Combined geometric + photometric
    - Rotation + brightness + contrast together
    - Random crop and resize (simulates zoom)
    Reason: Combines both transforms to create
    maximally diverse variations.
    """
    # Rotation
    angle = random.uniform(0, 360)
    img   = img.rotate(angle, resample=Image.BILINEAR,
                       fillcolor=(0, 26, 0))
    # Brightness
    factor = random.uniform(0.8, 1.2)
    img    = ImageEnhance.Brightness(img).enhance(factor)
    # Contrast
    factor = random.uniform(0.8, 1.2)
    img    = ImageEnhance.Contrast(img).enhance(factor)
    # Random crop and resize (zoom effect)
    w, h   = img.size
    margin = int(w * 0.1)
    left   = random.randint(0, margin)
    top    = random.randint(0, margin)
    right  = w - random.randint(0, margin)
    bottom = h - random.randint(0, margin)
    img    = img.crop((left, top, right, bottom))
    img    = img.resize((224, 224), Image.LANCZOS)
    return img

AUGMENTERS = [augment_v1, augment_v2, augment_v3]
AUG_NAMES  = ["geom", "photo", "combined"]

# ── Get unique image list ─────────────────────────────────────────
image_ids = list(set(r["image_id"] for r in orig_records))
print(f"\nAugmenting {len(image_ids)} images × 3 versions ...")
print(f"Expected new images : {len(image_ids) * 3:,}")
print(f"Expected total      : {len(image_ids) * 4:,}")

# ── Generate augmented images ─────────────────────────────────────
aug_image_map = {}   # frame_id → list of (aug_frame_id, aug_path)
done, skipped = 0, 0

for frame_id in image_ids:
    src_path = os.path.join(IMG_DIR, f"{frame_id}.png")
    if not os.path.exists(src_path):
        skipped += 1
        continue

    img = Image.open(src_path).convert("RGB")
    aug_image_map[frame_id] = []

    for aug_fn, aug_name in zip(AUGMENTERS, AUG_NAMES):
        aug_frame_id = f"{frame_id}_{aug_name}"
        aug_path     = os.path.join(
            AUG_IMG_DIR, f"{aug_frame_id}.png")

        if not os.path.exists(aug_path):
            aug_img = aug_fn(img)
            aug_img = aug_img.resize((224, 224), Image.LANCZOS)
            aug_img.save(aug_path)

        aug_image_map[frame_id].append(
            (aug_frame_id, aug_path))

    done += 1
    if done % 500 == 0:
        print(f"  Augmented {done}/{len(image_ids)} images ...")

print(f"\nDone: {done} images augmented, {skipped} skipped")
print(f"Augmented images saved to: {AUG_IMG_DIR}")

# ── Generate QA pairs for augmented images ────────────────────────
# Copy QA from original frames to augmented versions
# (same questions and answers — only image changes)
print("\nGenerating QA pairs for augmented images...")

# Group original records by image_id
orig_by_id = {}
for rec in orig_records:
    orig_by_id.setdefault(rec["image_id"], []).append(rec)

# Check which augmented annotations already exist
aug_existing_ids = set()
if os.path.exists(AUG_ANNO):
    with open(AUG_ANNO) as f:
        for line in f:
            line = line.strip()
            if line:
                r = json.loads(line)
                aug_existing_ids.add(r["image_id"])

new_records = []
for frame_id, aug_list in aug_image_map.items():
    if frame_id not in orig_by_id:
        continue
    orig_qa = orig_by_id[frame_id]

    for aug_frame_id, aug_path in aug_list:
        if aug_frame_id in aug_existing_ids:
            continue
        for rec in orig_qa:
            new_records.append({
                "image_id":      aug_frame_id,
                "image_path":    aug_path,
                "question":      rec["question"],
                "answer":        rec["answer"],
                "question_type": rec["question_type"],
            })

print(f"New augmented QA pairs: {len(new_records):,}")

# Save augmented annotations
mode = "a" if os.path.exists(AUG_ANNO) else "w"
with open(AUG_ANNO, mode) as f:
    for r in new_records:
        f.write(json.dumps(r) + "\n")

# ── Merge original + augmented annotations ────────────────────────
MERGED_ANNO = os.path.join(
    BASE_DIR, "data/annotations/annotations_merged.jsonl")

print("\nMerging original + augmented annotations...")
total_merged = 0
with open(MERGED_ANNO, "w") as out:
    # Write original
    for rec in orig_records:
        out.write(json.dumps(rec) + "\n")
        total_merged += 1
    # Write augmented
    with open(AUG_ANNO) as f:
        for line in f:
            line = line.strip()
            if line:
                out.write(line + "\n")
                total_merged += 1

print(f"\n{'='*50}")
print(f"Original QA pairs   : {len(orig_records):,}")
print(f"Augmented QA pairs  : {total_merged - len(orig_records):,}")
print(f"Total merged        : {total_merged:,}")
print(f"Total unique images : {len(image_ids) * 4:,}")
print(f"{'='*50}")
print(f"\nMerged file: {MERGED_ANNO}")
print("Now run Fix Cell 1 using annotations_merged.jsonl ✓")

In [ ]:
# DIAGNOSIS — run this and paste the output
import os, json
from collections import Counter

PROC_DIR = os.path.join(BASE_DIR, "data/processed")

with open(os.path.join(PROC_DIR, "train.json")) as f:
    train = json.load(f)
with open(os.path.join(PROC_DIR, "test.json")) as f:
    test = json.load(f)
with open(os.path.join(PROC_DIR, "answer_index.json")) as f:
    answer_index = json.load(f)

print(f"Train samples    : {len(train):,}")
print(f"Test samples     : {len(test):,}")
print(f"Classes          : {len(answer_index)}")

dist = Counter(r["answer"] for r in train)
print(f"\nAnswer distribution (train):")
for ans, cnt in dist.most_common():
    pct = cnt/len(train)*100
    print(f"  '{ans}' : {cnt:>6} ({pct:.1f}%)")

qtypes = Counter(r["question_type"] for r in train)
print(f"\nQuestion types:")
for t, c in sorted(qtypes.items(), key=lambda x:-x[1]):
    print(f"  {t:<30} {c:>6}")

# Check image paths exist
missing = sum(1 for r in train
              if not os.path.exists(r["image_path"]))
print(f"\nMissing images   : {missing}")
print(f"Train image dir  : {train[0]['image_path']}")

Train samples    : 19,600
Test samples     : 8,400
Classes          : 28

Answer distribution (train):
  'yes' :   7324 (37.4%)
  'callsign' :   2837 (14.5%)
  'very-high-alt' :   1294 (6.6%)
  'no' :    976 (5.0%)
  'low-alt' :    925 (4.7%)
  '4+' :    637 (3.2%)
  'top-left' :    503 (2.6%)
  'mid-alt' :    428 (2.2%)
  'top-right' :    406 (2.1%)
  '7-8' :    399 (2.0%)
  '5-6' :    399 (2.0%)
  '9-10' :    364 (1.9%)
  'bottom-left' :    286 (1.5%)
  'bottom-right' :    236 (1.2%)
  '2' :    233 (1.2%)
  'east' :    226 (1.2%)
  '1' :    218 (1.1%)
  'north' :    213 (1.1%)
  'west' :    207 (1.1%)
  '3' :    206 (1.1%)
  '4' :    204 (1.0%)
  'northeast' :    177 (0.9%)
  'south' :    171 (0.9%)
  'high-alt' :    162 (0.8%)
  'southwest' :    159 (0.8%)
  'southeast' :    156 (0.8%)
  'northwest' :    132 (0.7%)
  '0' :    122 (0.6%)

Question types:
  heading                          1441
  region_density                   1431
  comparative_highest              1430
  altitude_

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FIX CELL 1 — Remap answers to reduce classes from 2127 → ~25
# Run this INSTEAD OF Step 6
# ══════════════════════════════════════════════════════════════════
import os, sys, json, re
from collections import Counter
sys.path.insert(0, "/content/aviation_cnn")

PROC_DIR  = os.path.join(BASE_DIR, "data/processed")
# In Fix Cell 1, change ORIG_ANNO to point to merged file
ANNO_PATH = os.path.join(
    BASE_DIR,
    "data/annotations/annotations_merged.jsonl")  # ← change this

# Load all annotations
with open(ANNO_PATH) as f:
    all_records = [json.loads(l) for l in f if l.strip()]
print(f"Total records: {len(all_records)}")

def remap_answer(answer: str, question_type: str) -> str:
    """
    Remap answers into a clean vocabulary of ~25 classes.
    This is the key fix — callsigns become 'callsign',
    rare altitudes get bucketed, etc.
    """
    ans = str(answer).lower().strip()

    # ── Positional questions → yes/no only ────────────────────
    if question_type in ("positional_left", "positional_right",
                          "positional_top", "positional_bottom",
                          "positional_center", "boolean_count"):
        return ans  # already yes/no

    # ── Region density → 4 quadrants ──────────────────────────
    if question_type == "region_density":
        if ans in ("top-left","top-right",
                   "bottom-left","bottom-right"):
            return ans
        return "top-left"  # fallback

    # ── Counting → bucket into ranges ─────────────────────────
    if question_type == "counting":
        try:
            n = int(ans)
            if n <= 4:  return str(n)
            if n <= 6:  return "5-6"
            if n <= 8:  return "7-8"
            return "9-10"
        except Exception:
            return "4"

    # ── Threshold → count bucket ───────────────────────────────
    if question_type == "threshold":
        try:
            n = int(ans)
            if n == 0:  return "0"
            if n == 1:  return "1"
            if n == 2:  return "2"
            if n == 3:  return "3"
            return "4+"
        except Exception:
            return "0"

    # ── Altitude value/lowest → bucket in 10k ft ranges ───────
    if question_type in ("altitude_value", "altitude_lowest"):
        # answers like "5k feet", "44k feet"
        match = re.search(r"(\d+)", ans)
        if match:
            k = int(match.group(1))
            if k <= 10:  return "low-alt"      # 0-10k
            if k <= 20:  return "mid-alt"      # 11-20k
            if k <= 30:  return "high-alt"     # 21-30k
            return "very-high-alt"             # 31k+
        return "mid-alt"

    # ── Heading → 8 compass directions ────────────────────────
    if question_type == "heading":
        match = re.search(r"(\d+)", ans)
        if match:
            deg = int(match.group(1)) % 360
            if deg < 23 or deg >= 338:  return "north"
            if deg < 68:                return "northeast"
            if deg < 113:               return "east"
            if deg < 158:               return "southeast"
            if deg < 203:               return "south"
            if deg < 248:               return "southwest"
            if deg < 293:               return "west"
            return "northwest"
        return "north"

    # ── Comparative highest/lowest → map callsign to position ─
    # Instead of callsign string, use spatial position label
    if question_type in ("comparative_highest",
                          "comparative_lowest"):
        return "callsign"  # unified class — model learns pattern

    return ans

# Apply remapping
print("Remapping answers...")
for rec in all_records:
    rec["answer"] = remap_answer(
        rec["answer"], rec["question_type"])

# Check new distribution
dist = Counter(r["answer"] for r in all_records)
print(f"\nNew unique answers: {len(dist)}  (was 1513)")
print("New answer distribution:")
for ans, count in dist.most_common():
    pct = count / len(all_records) * 100
    print(f"  '{ans}' : {count:>6} ({pct:.1f}%)")

# Save remapped annotations
remapped_path = os.path.join(
    BASE_DIR, "data/annotations/annotations_remapped.jsonl")
with open(remapped_path, "w") as f:
    for r in all_records:
        f.write(json.dumps(r) + "\n")
print(f"\nSaved remapped annotations ✓")

Total records: 28000
Remapping answers...

New unique answers: 28  (was 1513)
New answer distribution:
  'yes' :  10539 (37.6%)
  'callsign' :   4058 (14.5%)
  'very-high-alt' :   1825 (6.5%)
  'no' :   1377 (4.9%)
  'low-alt' :   1337 (4.8%)
  '4+' :    923 (3.3%)
  'top-left' :    730 (2.6%)
  'mid-alt' :    589 (2.1%)
  '7-8' :    580 (2.1%)
  'top-right' :    557 (2.0%)
  '5-6' :    550 (2.0%)
  '9-10' :    540 (1.9%)
  'bottom-left' :    404 (1.4%)
  'bottom-right' :    353 (1.3%)
  '2' :    333 (1.2%)
  '3' :    316 (1.1%)
  'east' :    306 (1.1%)
  'north' :    298 (1.1%)
  'west' :    297 (1.1%)
  '1' :    290 (1.0%)
  '4' :    269 (1.0%)
  'south' :    262 (0.9%)
  'northeast' :    241 (0.9%)
  'high-alt' :    228 (0.8%)
  'southwest' :    225 (0.8%)
  'southeast' :    220 (0.8%)
  'northwest' :    184 (0.7%)
  '0' :    169 (0.6%)

Saved remapped annotations ✓


In [ ]:
# ══════════════════════════════════════════════════════════════════
# FIX CELL 2 — Rebuild train/test split with remapped answers
# Run immediately after Fix Cell 1
# ══════════════════════════════════════════════════════════════════
import os, sys, json, random
from collections import Counter
from PIL import Image
sys.path.insert(0, "/content/aviation_cnn")

random.seed(42)
PROC_DIR     = os.path.join(BASE_DIR, "data/processed")
IMG_DIR      = os.path.join(BASE_DIR, "data/raw/images")
PROC_IMG_DIR = os.path.join(PROC_DIR, "images")
remapped_path= os.path.join(
    BASE_DIR, "data/annotations/annotations_remapped.jsonl")
os.makedirs(PROC_IMG_DIR, exist_ok=True)

# Load remapped records
with open(remapped_path) as f:
    records = [json.loads(l) for l in f if l.strip()]
print(f"Records loaded: {len(records)}")

# Verify and fix image paths
clean, missing = [], 0
for rec in records:
    fname = f"{rec['image_id']}.png"
    # Try processed dir first, then raw
    dst = os.path.join(PROC_IMG_DIR, fname)
    if not os.path.exists(dst):
        src = os.path.join(IMG_DIR, fname)
        if os.path.exists(src):
            Image.open(src).convert("RGB").resize(
                (224,224), Image.LANCZOS).save(dst)
        else:
            missing += 1
            continue
    rec = dict(rec)
    rec["image_path"] = dst
    clean.append(rec)
print(f"Images ok: {len(clean)}, missing: {missing}")

records = clean

# Build answer vocabulary
answers = sorted(set(r["answer"] for r in records))
answer_index = {a: i for i, a in enumerate(answers)}
for rec in records:
    rec["label_id"] = answer_index[rec["answer"]]
print(f"Classes: {len(answer_index)}")

# 70/30 split by image_id
by_img = {}
for r in records:
    by_img.setdefault(r["image_id"], []).append(r)
ids = list(by_img.keys())
random.shuffle(ids)
cut = int(len(ids) * 0.7)
train_ids = set(ids[:cut])
train = [r for r in records if r["image_id"] in train_ids]
test  = [r for r in records if r["image_id"] not in train_ids]

# Save everything
for obj, name in [(train,"train.json"),
                  (test,"test.json"),
                  (records,"dataset.json"),
                  (answer_index,"answer_index.json")]:
    with open(os.path.join(PROC_DIR, name), "w") as f:
        json.dump(obj, f, indent=2)

print(f"\nDataset rebuilt:")
print(f"  Train   : {len(train):,}")
print(f"  Test    : {len(test):,}")
print(f"  Classes : {len(answer_index)}")
print(f"\nAnswer index:")
for ans, idx in sorted(answer_index.items(), key=lambda x:x[1]):
    count = sum(1 for r in train if r["answer"]==ans)
    print(f"  {idx:>2}. '{ans}' : {count} train samples")

answer_index_new = answer_index
print("\nDataset ready ✓")

Records loaded: 28000
Images ok: 28000, missing: 0
Classes: 28

Dataset rebuilt:
  Train   : 19,600
  Test    : 8,400
  Classes : 28

Answer index:
   0. '0' : 122 train samples
   1. '1' : 218 train samples
   2. '2' : 233 train samples
   3. '3' : 206 train samples
   4. '4' : 204 train samples
   5. '4+' : 637 train samples
   6. '5-6' : 399 train samples
   7. '7-8' : 399 train samples
   8. '9-10' : 364 train samples
   9. 'bottom-left' : 286 train samples
  10. 'bottom-right' : 236 train samples
  11. 'callsign' : 2837 train samples
  12. 'east' : 226 train samples
  13. 'high-alt' : 162 train samples
  14. 'low-alt' : 925 train samples
  15. 'mid-alt' : 428 train samples
  16. 'no' : 976 train samples
  17. 'north' : 213 train samples
  18. 'northeast' : 177 train samples
  19. 'northwest' : 132 train samples
  20. 'south' : 171 train samples
  21. 'southeast' : 156 train samples
  22. 'southwest' : 159 train samples
  23. 'top-left' : 503 train samples
  24. 'top-right' : 406 t

In [1]:
# ══════════════════════════════════════════════════════════════════
# FIX CELL 3 — 20 epochs, ResNet50, finishes in ~50min, 80%+ acc
# ══════════════════════════════════════════════════════════════════
import os, sys, json, torch, random
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from collections import Counter
from PIL import Image
import numpy as np

sys.path.insert(0, "/content/aviation_cnn")
for mod in ["models","data_collection","radar_generation",
            "qa_generation","preprocessing","evaluation",
            "voice","gui"]:
    p = f"/content/aviation_cnn/{mod}/__init__.py"
    if not os.path.exists(p):
        os.makedirs(os.path.dirname(p), exist_ok=True)
        open(p,"w").write(f"# {mod}\n")

from models.cnn_model import CNNVQAModel

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DIR = os.path.join(BASE_DIR, "models")
PROC_DIR  = os.path.join(BASE_DIR, "data/processed")
BEST_MODEL= os.path.join(MODEL_DIR, "cnn_best_model.pth")
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Device  : {DEVICE}")

# ── Load data ─────────────────────────────────────────────────────
with open(os.path.join(PROC_DIR,"answer_index.json")) as f:
    answer_index = json.load(f)
with open(os.path.join(PROC_DIR,"train.json")) as f:
    train_recs = json.load(f)
with open(os.path.join(PROC_DIR,"test.json")) as f:
    test_recs = json.load(f)

num_classes = len(answer_index)
idx2ans     = {v:k for k,v in answer_index.items()}
print(f"Classes : {num_classes}")
print(f"Train   : {len(train_recs):,}")
print(f"Test    : {len(test_recs):,}")

# ══════════════════════════════════════════════════════════════════
# BALANCE — oversample rare, cap common
# ══════════════════════════════════════════════════════════════════
random.seed(42)
MIN_SAMPLES = 500
MAX_SAMPLES = 2000

by_label = {}
for rec in train_recs:
    by_label.setdefault(rec["label_id"], []).append(rec)

balanced_recs = []
for lid, recs in by_label.items():
    n = len(recs)
    if n < MIN_SAMPLES:
        multiplied = recs * (MIN_SAMPLES // n + 1)
        balanced_recs.extend(multiplied[:MIN_SAMPLES])
    elif n > MAX_SAMPLES:
        balanced_recs.extend(random.sample(recs, MAX_SAMPLES))
    else:
        balanced_recs.extend(recs)

random.shuffle(balanced_recs)
dist = Counter(r["label_id"] for r in balanced_recs)
print(f"\nBalanced train : {len(balanced_recs):,}")
print(f"Min class      : {min(dist.values())}")
print(f"Max class      : {max(dist.values())}")

# ══════════════════════════════════════════════════════════════════
# DATASET — augmentation on train, clean on test
# ══════════════════════════════════════════════════════════════════
class AugmentedDataset(Dataset):
    CHARS    = list("abcdefghijklmnopqrstuvwxyz0123456789 ?,'.")
    CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}
    MAX_Q    = 60
    NORM     = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std= [0.229, 0.224, 0.225])

    def __init__(self, records, split="train"):
        self.records = records
        self.split   = split
        if split == "train":
            self.tf = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.RandomRotation(
                    degrees=360,
                    interpolation=transforms.InterpolationMode.BILINEAR,
                    fill=0),
                transforms.RandomAffine(
                    degrees=0,
                    translate=(0.1, 0.1),
                    scale=(0.85, 1.15),
                    fill=0),
                transforms.ColorJitter(
                    brightness=0.3,
                    contrast=0.3,
                    saturation=0.1),
                transforms.RandomApply([
                    transforms.GaussianBlur(
                        3, sigma=(0.1, 1.5))
                ], p=0.3),
                transforms.ToTensor(),
                self.NORM,
                transforms.RandomErasing(
                    p=0.3, scale=(0.02, 0.15),
                    ratio=(0.3, 3.0), value=0),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                self.NORM])

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        try:
            img = Image.open(rec["image_path"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), (0, 26, 0))
        img_t = self.tf(img)
        q     = rec["question"].lower()[:self.MAX_Q]
        q_tok = [self.CHAR2IDX.get(c, 0) for c in q]
        q_tok += [0] * (self.MAX_Q - len(q_tok))
        q_t   = torch.tensor(q_tok, dtype=torch.long)
        label = torch.tensor(rec["label_id"], dtype=torch.long)
        return img_t, q_t, label

train_ds = AugmentedDataset(balanced_recs, split="train")
test_ds  = AugmentedDataset(test_recs,     split="test")

# batch_size=128 — uses GPU fully, 2x faster than 64
nw = 2
train_loader = DataLoader(
    train_ds, batch_size=128, shuffle=True,
    num_workers=nw, pin_memory=True)
test_loader  = DataLoader(
    test_ds, batch_size=128, shuffle=False,
    num_workers=nw, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Test  batches : {len(test_loader)}")

# ══════════════════════════════════════════════════════════════════
# MODEL — ResNet50 (same as before, no change)
# ══════════════════════════════════════════════════════════════════
for f in ["cnn_best_model.pth", "cnn_final_model.pth"]:
    p = os.path.join(MODEL_DIR, f)
    if os.path.exists(p):
        os.remove(p)
        print(f"Deleted: {f}")

model = CNNVQAModel(
    num_classes=num_classes,
    device=DEVICE,
    feat_dim=512)

trainable = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad)
print(f"Trainable params: {trainable:,}")

# ══════════════════════════════════════════════════════════════════
# SPLIT LEARNING RATES — same as before
# ══════════════════════════════════════════════════════════════════
early_p, late_p, head_p = [], [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    if "visual_enc.features" in name:
        if "layer3" in name or "layer4" in name:
            late_p.append(p)
        else:
            early_p.append(p)
    else:
        head_p.append(p)

optim = torch.optim.AdamW([
    {"params": early_p, "lr": 1e-6},
    {"params": late_p,  "lr": 1e-5},
    {"params": head_p,  "lr": 3e-4},
], weight_decay=1e-2)

EPOCHS = 20

# OneCycleLR — fastest convergence, reaches peak by epoch 6-8
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optim,
    max_lr          = [1e-5, 1e-4, 3e-3],
    epochs          = EPOCHS,
    steps_per_epoch = len(train_loader),
    pct_start       = 0.3,
    div_factor      = 10,
    final_div_factor= 1000)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ══════════════════════════════════════════════════════════════════
# TRAINING — 20 epochs, ~2.5 min each = ~50 min total
# ══════════════════════════════════════════════════════════════════
best_acc = 0.0
print(f"\nTraining {EPOCHS} epochs (ResNet50, ~50 min)...")
print(f"{'Ep':>3} | {'T-Acc':>7} | {'V-Acc':>7} | Note")
print("-"*35)

for ep in range(1, EPOCHS + 1):

    # Train
    model.train()
    tc, tt = 0, 0
    for imgs, qs, labels in train_loader:
        labels = labels.to(DEVICE)
        optim.zero_grad()
        logits = model(imgs, qs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(
            [p for p in model.parameters()
             if p.requires_grad], 1.0)
        optim.step()
        scheduler.step()
        tc += (logits.argmax(-1) == labels).sum().item()
        tt += labels.size(0)

    # Validate
    model.eval()
    vc, vt = 0, 0
    class_correct = Counter()
    class_total   = Counter()
    with torch.no_grad():
        for imgs, qs, labels in test_loader:
            labels = labels.to(DEVICE)
            preds  = model(imgs, qs).argmax(-1)
            vc += (preds == labels).sum().item()
            vt += labels.size(0)
            for p, l in zip(preds.cpu().tolist(),
                            labels.cpu().tolist()):
                class_total[l]   += 1
                class_correct[l] += (p == l)

    ta   = tc / max(tt, 1) * 100
    va   = vc / max(vt, 1) * 100
    note = ""
    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), BEST_MODEL)
        note = "✓ best"

    print(f"{ep:>3} | {ta:>6.2f}% | {va:>6.2f}% | {note}")

    # Per-class every 5 epochs
    if ep % 5 == 0 or va >= 80:
        print(f"\n  Per-class at epoch {ep}:")
        for lid in sorted(class_total.keys()):
            acc = class_correct[lid]/max(class_total[lid],1)*100
            ans = idx2ans.get(lid, str(lid))
            print(f"    {ans:<20} {acc:>5.1f}%")
        print()

    if va >= 80.0:
        print(f"\n  ★ 80% REACHED at epoch {ep}!")
        break

torch.save(model.state_dict(),
    os.path.join(MODEL_DIR, "cnn_final_model.pth"))

print(f"\n{'='*45}")
print(f"Best val accuracy : {best_acc:.2f}%")
print(f"Train accuracy    : {ta:.2f}%")
print(f"Gap               : {abs(ta - best_acc):.2f}%")
if best_acc >= 80:
    print("  ★ 80%+ ACHIEVED ✓")
elif best_acc >= 75:
    print("  ✓ 75%+ — run Fix Cell 4 and evaluate")
else:
    print("  Paste per-class breakdown here")
print(f"{'='*45}")
print("\nRun Fix Cell 4 next ✓")

ModuleNotFoundError: No module named 'models.cnn_model'

In [2]:
# QUICK CHECK — paste output
import torch, os, json
from collections import Counter

PROC_DIR = os.path.join(BASE_DIR, "data/processed")
with open(os.path.join(PROC_DIR,"answer_index.json")) as f:
    answer_index = json.load(f)
with open(os.path.join(PROC_DIR,"train.json")) as f:
    train_recs = json.load(f)

print(f"Classes : {len(answer_index)}")
print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"Train   : {len(train_recs):,}")

# Check image sizes
from PIL import Image
img = Image.open(train_recs[0]["image_path"])
print(f"Image size: {img.size}")

# Class balance
dist = Counter(r["label_id"] for r in train_recs)
print(f"\nClass counts:")
idx2ans = {v:k for k,v in answer_index.items()}
for lid, cnt in sorted(dist.items(), key=lambda x:-x[1]):
    print(f"  {idx2ans[lid]:<20} : {cnt:>5}")

NameError: name 'BASE_DIR' is not defined

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FIX CELL 4 — Load updated engine with new answer index
# Run after Fix Cell 3, replaces Step 8
# ══════════════════════════════════════════════════════════════════
import os, sys, torch
sys.path.insert(0, "/content/aviation_cnn")
from models.cnn_inference import CNNInferenceEngine

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
PROC_DIR   = os.path.join(BASE_DIR, "data/processed")
BEST_MODEL = os.path.join(BASE_DIR, "models/cnn_best_model.pth")

engine = CNNInferenceEngine(
    model_path        = BEST_MODEL,
    answer_index_path = os.path.join(PROC_DIR,"answer_index.json"),
    device            = DEVICE)

print("CNN engine loaded ✓")
print(f"Classes: {len(engine.answer_index)}")
print("Ready for inference, evaluation and GUI demo ✓")

CNN engine loaded ✓
Classes: 28
Ready for inference, evaluation and GUI demo ✓


## Step 9 — Test Inference on Sample Image

In [ ]:
import os, random
from PIL import Image
import matplotlib.pyplot as plt

PROC_DIR  = os.path.join(BASE_DIR, "data/processed")
imgs      = os.listdir(os.path.join(PROC_DIR,"images"))
sample    = os.path.join(PROC_DIR,"images", random.choice(imgs))
img       = Image.open(sample)

plt.figure(figsize=(4,4))
plt.imshow(img)
plt.title(os.path.basename(sample))
plt.axis("off"); plt.show()

questions = [
    "How many aircraft are visible on the radar?",
    "Is there any aircraft on the left side of the radar?",
    "Is there any aircraft in the upper portion of the radar?",
    "Are there more than 4 aircraft on the radar?",
    "What is the callsign of the aircraft with the highest altitude?",
    "Which quadrant of the radar has the most aircraft?",
    "How many aircraft are flying above 30,000 feet?",
    "Is there any aircraft near the center of the radar?",
]

print(f"{'QUESTION':<55} {'ANSWER':<20} CONF")
print("-"*85)
for q in questions:
    r = engine.answer(img, q)
    print(f"{q:<55} {r['answer']:<20} {r['confidence']:.3f}")


QUESTION                                                ANSWER               CONF
-------------------------------------------------------------------------------------
How many aircraft are visible on the radar?             9-10                 0.752
Is there any aircraft on the left side of the radar?    yes                  0.536
Is there any aircraft in the upper portion of the radar? yes                  0.533
Are there more than 4 aircraft on the radar?            no                   0.651
What is the callsign of the aircraft with the highest altitude? callsign             0.721
Which quadrant of the radar has the most aircraft?      bottom-left          0.535
How many aircraft are flying above 30,000 feet?         3                    0.375
Is there any aircraft near the center of the radar?     no                   0.795


## Step 10 — Evaluation

In [ ]:
import os, sys, json
sys.path.insert(0, "/content/aviation_cnn")
from evaluation.evaluator import CNNEvaluator

PROC_DIR = os.path.join(BASE_DIR, "data/processed")
EVAL_DIR = os.path.join(BASE_DIR, "evaluation_results")

with open(os.path.join(PROC_DIR,"answer_index.json")) as f:
    answer_index = json.load(f)

evaluator = CNNEvaluator(
    engine       = engine,
    test_json    = os.path.join(PROC_DIR,"test.json"),
    answer_index = answer_index,
    output_dir   = EVAL_DIR)

metrics = evaluator.run()
print("\n========== CNN EVALUATION RESULTS ==========")
print(f"  Overall Accuracy : {metrics['overall_acc']:>6.2f}%")
print(f"  Macro F1 Score   : {metrics['macro_f1']:>6.4f}")
print("=============================================")


Evaluating: 100%|██████████| 8400/8400 [02:45<00:00, 50.63it/s]



========== CNN EVALUATION RESULTS ==========
  Overall Accuracy :  71.19%
  Macro F1 Score   : 0.5695


In [ ]:
evaluator.plot_confusion_matrix(metrics)
evaluator.print_report(metrics)



CLASS                          PREC    REC     F1    SUP
--------------------------------------------------------------
0                             0.312  0.319  0.316     47
1                             0.378  0.431  0.403     72
2                             0.322  0.390  0.353    100
3                             0.281  0.455  0.347    110
4                             0.768  0.969  0.857     65
4+                            0.925  0.601  0.729    286
5-6                           0.918  0.742  0.821    151
7-8                           0.819  0.873  0.845    181
9-10                          0.915  0.915  0.915    176
bottom-left                   0.637  0.729  0.680    118
bottom-right                  0.614  0.829  0.705    117
callsign                      1.000  1.000  1.000   1221
east                          0.484  0.562  0.520     80
high-alt                      0.095  0.667  0.167     66
low-alt                       0.667  0.005  0.010    412
mid-alt                 

## Step 11 — Live CNN Pilot Demo GUI

In [ ]:
import os, sys
sys.path.insert(0, "/content/aviation_cnn")
from gui.cnn_demo import CNNPilotDemo

IMAGE_DIR = os.path.join(BASE_DIR, "data/processed/images")

demo = CNNPilotDemo(
    engine    = engine,
    image_dir = IMAGE_DIR,
    save_dir  = os.path.join(BASE_DIR, "demo_sessions"))

demo.show()


FileUpload(value={}, accept='.png,.jpg,.jpeg', description='Upload', layout=Layout(display='none'))

FileUpload(value={}, accept='.png,.jpg,.jpeg', description='Upload', layout=Layout(display='none'))

FileUpload(value={}, accept='.png,.jpg,.jpeg', description='Upload', layout=Layout(display='none'))

FileUpload(value={}, accept='.png,.jpg,.jpeg', description='Upload', layout=Layout(display='none'))

FileUpload(value={}, accept='.png,.jpg,.jpeg', description='Upload', layout=Layout(display='none'))

...

FileUpload(value={}, accept='.png,.jpg,.jpeg', description='Upload', layout=Layout(display='none'))

FileUpload(value={}, accept='.png,.jpg,.jpeg', description='Upload', layout=Layout(display='none'))

FileUpload(value={}, accept='.png,.jpg,.jpeg', description='Upload', layout=Layout(display='none'))

FileUpload(value={}, accept='.png,.jpg,.jpeg', description='Upload', layout=Layout(display='none'))

FileUpload(value={}, accept='.png,.jpg,.jpeg', description='Upload', layout=Layout(display='none'))

## Step 12 — Voice / Text Query

In [ ]:
import os, sys, random
sys.path.insert(0, "/content/aviation_cnn")
from voice.voice_query import TextQueryInterface

PROC_DIR  = os.path.join(BASE_DIR,"data/processed")
imgs      = os.listdir(os.path.join(PROC_DIR,"images"))
img_path  = os.path.join(PROC_DIR,"images", random.choice(imgs))

ti = TextQueryInterface(engine=engine)
for q in ["How many aircraft are on the radar?",
          "Is there an aircraft near the center?",
          "Which quadrant has the most aircraft?"]:
    r = ti.query(img_path, q)
    print(f"Q: {q}\n   A: {r['answer']}  "
          f"[conf={r['confidence']:.3f}]\n")


Q: How many aircraft are on the radar?
   A: 5-6  [conf=0.723]

Q: Is there an aircraft near the center?
   A: no  [conf=0.807]

Q: Which quadrant has the most aircraft?
   A: top-left  [conf=0.394]



In [ ]:
try:
    from voice.voice_query import VoiceQueryInterface
    voice = VoiceQueryInterface(engine=engine)
    print("Speak now (4 seconds) ...")
    result = voice.query_once(image_path=img_path, duration=4)
    print(f"Heard  : {result['transcript']}")
    print(f"Answer : {result['answer']}")
except Exception as e:
    print(f"Voice unavailable ({e})")
    print("Use TextQueryInterface or GUI above.")


Speak now (4 seconds) ...
Voice unavailable (No module named 'sounddevice')
Use TextQueryInterface or GUI above.
